In [25]:
import os
import json
import re
import pandas as pd
import numpy as np
import pprint

from utils import exact_match_results, relax_match_results


* model_folder: output folder without similarity algorithm
* model_sim_folder: output folder after running similarity algorithm

In [26]:
# model_folder = 'Output_gemma4_26b_no_sim'
# model_folder = "Output_llama3.1_8b_no_sim"
model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_31b_no_sim"

# model_sim_folder = 'Output_llama3.1_8b_sim'
# model_sim_folder = "Output_gemma4_31b_sim"
model_sim_folder = "Output_mistral-small_24b_sim"
# model_sim_folder = "Output_gemma4_26b_sim"

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

In [27]:
# # Checking if there's any null output:
# for sample in os.listdir(model_folder):
#     with open(f"{model_folder}/{sample}", "r") as f:
#         data = json.load(f)
#         if not data:
#             print(f"Sample {sample} has null output.")


# Identify all rephrased entities

In [28]:
preds_dict_od_all = []
# preds_dict_od_all_final = []
preds_dict_all_temp = []
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        # print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()
        preds_dict_all_temp.append(preds_dict)
        preds_dict_processed = []
        preds_dict_od = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:

                    preds_dict_od.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                    preds_dict_od_all.append({'sample':sample,'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
        
        # for item in preds_dict_od_all:
        #     if item not in preds_dict_od_all_final:
        #         preds_dict_od_all_final.append(item)
        # with open(f"{model_attribution_folder}/{sample}", 'w') as f:
        #     json.dump(preds_dict_od, f, indent=2)

print(len(preds_dict_od_all))
pprint.pprint(preds_dict_od_all)

50
[{'end': 0,
  'entity': 'a strictured bile duct',
  'label': 'OD',
  'sample': 'sample_38_20210610044413_Hayden',
  'start': 0},
 {'end': 0,
  'entity': 'abdominal pain',
  'label': 'OD',
  'sample': 'sample_1765',
  'start': 0},
 {'end': 0,
  'entity': 'the pelvis',
  'label': 'OD',
  'sample': 'sample_50_20210604143818_Isra',
  'start': 0},
 {'end': 0,
  'entity': 'urinary infection',
  'label': 'OD',
  'sample': 'sample_76_20210112211704',
  'start': 0},
 {'end': 0,
  'entity': 'suicidal ideations',
  'label': 'OD',
  'sample': 'sample_36_20210610044412_Hayden',
  'start': 0},
 {'end': 0,
  'entity': 'Femoral pulses',
  'label': 'OD',
  'sample': 'sample_30_20210610044412_Isra',
  'start': 0},
 {'end': 0,
  'entity': 'upper extremity',
  'label': 'OD',
  'sample': 'sample_30_20210610044412_Isra',
  'start': 0},
 {'end': 0,
  'entity': 'suicidal ideation',
  'label': 'OD',
  'sample': 'sample_30_20210610044412_Isra',
  'start': 0},
 {'end': 0,
  'entity': 'central canal narrowing'

# Similarity algorithm

### Original Method

In [29]:
from sentence_transformers import SentenceTransformer, util
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f"Using device: {device}")

for sample in os.listdir(model_folder):
    # if sample == 'sample_134_20210112211711':
    #Read original text
    #Read label dict
        print(f"processing: {sample}")

        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)
            
        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            org_preds_json = json.load(f)

        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()
        
        #Find the messed up tokens in the output
        preds_dict = []
        messup_pred_tokens = []
        for ent in org_preds_json:
            for phrase, label in ent.items():
                matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    messup_pred_tokens.append(phrase)
        
  
        #Clean up the note
        clean_text = re.sub(r'[.,:&]', '', text)
        clean_original_tokens = clean_text.split()
        n = len(clean_original_tokens)
        print(f"length of the text: {n}")

        
        # Attribution algorithm
        unique_messup_pred_tokens = list(dict.fromkeys(messup_pred_tokens))
        max_target_length = max((len(token.split()) for token in unique_messup_pred_tokens if token.strip()), default=0)
        target_lengths = range(1, max_target_length*5)
        
        spans = []
        seen_spans = set()
        for i in range(n):
            remaining = n - i
            for width in target_lengths:
                if width > remaining:
                    break
                span = ' '.join(clean_original_tokens[i:i + width])
                if span not in seen_spans:
                    seen_spans.add(span)
                    spans.append(span)

        if spans and unique_messup_pred_tokens:
            spans_emb = model.encode(
                spans,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,
            
            ).to(device)

            target_emb = model.encode(
                unique_messup_pred_tokens,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,

            ).to(device)

            sim = util.cos_sim(target_emb, spans_emb)
            top_indices = sim.argmax(dim=1).tolist()
            resolved_tokens = [spans[idx] for idx in top_indices]
            resolved_mapping = dict(zip(unique_messup_pred_tokens, resolved_tokens))
            alg_pred_tokens = [resolved_mapping[token] for token in messup_pred_tokens]
        else:
            alg_pred_tokens = []

        #Write back the output
        mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))
        original = []
        updated = []

        for record in org_preds_json:
            original_record = {}
            updated_record = {}

            for key, value in record.items():

                if key in mapping:
                    original_record[key] = "false positive"
                    updated_record[mapping[key]] = value
                    
                else:
                    original_record[key] = value
                    updated_record[key] = value

            original.append(original_record)
            updated.append(updated_record)
        
        with open(f"{model_sim_folder}/{sample}", 'w') as f:
            json.dump(updated, f, indent=2)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17085.75it/s]


Using device: cuda
processing: sample_38_20210610044413_Hayden
length of the text: 540
processing: sample_1765
length of the text: 368
processing: sample_160_20210112211715
length of the text: 575
processing: sample_1231
length of the text: 223
processing: sample_60_20210604143820_Isra
length of the text: 466
processing: sample_50_20210604143818_Isra
length of the text: 602
processing: sample_73_20210610044426_Booma
length of the text: 377
processing: sample_154_20210112211713
length of the text: 383
processing: sample_76_20210112211704
length of the text: 241
processing: sample_364
length of the text: 556
processing: sample_963
length of the text: 366
processing: sample_166_20210112211715
length of the text: 749
processing: sample_85_20210112211705
length of the text: 457
processing: sample_128_20210112211711
length of the text: 665
processing: sample_5_20210604143806_Hayden
length of the text: 406
processing: sample_2765
length of the text: 243
processing: sample_2762
length of the t

### New Method

In [30]:
# def find_max_sim_containing_token(model, token_idx, token_sim, ai_output_encode, tokens, checked, max_span):
#   l = token_idx
#   r = token_idx + 1

#   lb = l
#   rb = r

#   n = len(tokens)

#   max_sim = token_sim
#   txt = tokens[token_idx]
#   max_sim_text = txt

#   # check right-hand side
#   while (r<n) and (not checked[r]):
#     txt += f' {tokens[r]}'
#     r += 1
#     text_encode = model.encode(txt,
#                               batch_size=128,
#                               convert_to_tensor=True,
#                               normalize_embeddings=True,
#                                ).to(device)
#     sim = util.cos_sim(text_encode, ai_output_encode).item()
#     # print(f'{txt}: {sim}')
#     if sim > max_sim:
#       max_sim_text = txt
#       max_sim = sim
#       rb = r
#     if r-(token_idx + 1) >= max_span:
#       break

#   txt = max_sim_text

#   # check left-hand side
#   while (l>0) and (not checked[l-1]):
#     l -= 1
#     txt = f'{tokens[l]} ' + txt
#     text_encode = model.encode(txt,
#                               batch_size=128,
#                               convert_to_tensor=True,
#                               normalize_embeddings=True,
#                                ).to(device)
#     sim = util.cos_sim(text_encode, ai_output_encode).item()
#     # print(f'{txt}: {sim}')
#     if sim > max_sim:
#       max_sim_text = txt
#       max_sim = sim
#       lb = l
#     if token_idx-l >= max_span:
#       break

#   return max_sim_text, max_sim

# def find_max_sim(model, ai_output, tokens, max_span=1000):
#   n = len(tokens)
#   checked = [False] * n
#   ai_output_encode = model.encode(ai_output,
#                               batch_size=128,
#                               convert_to_tensor=True,
#                               normalize_embeddings=True,
#                                ).to(device)

#   sims = []
#   for i, token in enumerate(tokens):
#     text_encode = model.encode(token,
#                               batch_size=128,
#                               convert_to_tensor=True,
#                               normalize_embeddings=True,
#                                ).to(device)
#     sim = util.cos_sim(text_encode, ai_output_encode).item()
#     sims.append((i, sim))

#   sims.sort(key=lambda x: x[1], reverse=True)
#   # print(sims)
#   # print("---------------------------------------")

#   best_match = ""
#   best_score = float("-inf")

#   for i in sims:
#     y_max = 0
#     x, y = find_max_sim_containing_token(model, i[0], i[1], ai_output_encode, tokens, checked, max_span)
#     checked[i[0]] = True
#     if y > best_score:
#       best_score = y
#       best_match = x
#     # print(f"Starting from '{tokens[i[0]]}': {x}: {y}")
#     # print("---------------------------------------")
#   return best_match

# from sentence_transformers import SentenceTransformer, util
# import torch

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
# print(f"Using device: {device}")

# for sample in os.listdir(model_folder):
#     # if sample == 'sample_134_20210112211711':
#       print(f"processing: {sample}")

#       with open(f'label/{sample}.json', 'r') as f:
#           label_dict = json.load(f)
          
#       #Read output
#       with open(f"{model_folder}/{sample}", 'r') as f:
#           org_preds_json = json.load(f)

#       #Read original text
#       with open(f'notes/{sample}.txt', 'r') as file:
#           text = file.read()

#       #Find the messed up tokens in the output
#       preds_dict = []
#       messup_pred_tokens = []
#       for ent in org_preds_json:
#           for phrase, label in ent.items():
#               matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
#               if matches:
#                   for match in matches:
#                       # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
#                       preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
#               else:
#                   messup_pred_tokens.append(phrase)
    
      
#       # Clean up the note
#       clean_text = re.sub(r'[.,:&]', '', text)
#       clean_original_tokens = clean_text.split()
#       n = len(clean_original_tokens)
#       print(f"length of the text: {n}")

#       #Attribution algorithm
#       unique_messup_pred_tokens = list(dict.fromkeys(messup_pred_tokens))
#       resolved_mapping = {}

#       for ai_output in unique_messup_pred_tokens:
#           best_match = find_max_sim(
#               model,
#               ai_output,
#               clean_original_tokens,
#               max_span=(len(ai_output.split()) * 5 - 1))
          
#           resolved_mapping[ai_output] = best_match

#       alg_pred_tokens = [resolved_mapping[token] for token in messup_pred_tokens]

#       #Write back the output
#       mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))
#       original = []
#       updated = []

#       for record in org_preds_json:
#           original_record = {}
#           updated_record = {}

#           for key, value in record.items():

#               if key in mapping:
#                   original_record[key] = "false positive"
#                   updated_record[mapping[key]] = value
                  
#               else:
#                   original_record[key] = value
#                   updated_record[key] = value

#           original.append(original_record)
#           updated.append(updated_record)
      
#       with open(f"{model_sim_folder}/{sample}", 'w') as f:
#           json.dump(updated, f, indent=2)




# Evaluation - Attribution Results

In [ ]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_sim_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_sim_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + fp_rm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + fp_em + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall):.4f}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall):.4f}")

evaluating sample_38_20210610044413_Hayden:
[{'entity': 'CT scan', 'label': 'TEST', 'start': 290, 'end': 297}, {'entity': 'CT scan', 'label': 'TEST', 'start': 732, 'end': 739}, {'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 330, 'end': 345}, {'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 781, 'end': 796}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 360, 'end': 371}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 811, 'end': 822}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 3226, 'end': 3237}, {'entity': 'ERCP', 'label': 'TEST', 'start': 3009, 'end': 3013}, {'entity': 'placement of a stent', 'label': 'TREATMENT', 'start': 1016, 'end': 1036}, {'entity': '10-frenchx9 cm stent', 'label': 'TREATMENT', 'start': 1117, 'end': 1137}, {'entity': 'bilirubin', 'label': 'TEST', 'start': 1275, 'end': 1284}, {'entity': 'bilirubin', 'label': 'TEST', 'start': 1845, 'end': 1854}, {'entity': 'bilirubin', 'label': 'TEST', 'start': 3077, 'end': 3086}, {

# Evaluation - Results before attribution

In [ ]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + fp_rm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + fp_em + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall):.4f}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall):.4f}")

evaluating sample_38_20210610044413_Hayden:
[{'entity': 'CT scan', 'label': 'TEST', 'start': 290, 'end': 297}, {'entity': 'CT scan', 'label': 'TEST', 'start': 732, 'end': 739}, {'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 330, 'end': 345}, {'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 781, 'end': 796}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 360, 'end': 371}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 811, 'end': 822}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 3226, 'end': 3237}, {'entity': 'ERCP', 'label': 'TEST', 'start': 3009, 'end': 3013}, {'entity': 'placement of a stent', 'label': 'TREATMENT', 'start': 1016, 'end': 1036}, {'entity': 'a strictured bile duct', 'label': 'OD', 'start': 0, 'end': 0}, {'entity': '10-frenchx9 cm stent', 'label': 'TREATMENT', 'start': 1117, 'end': 1137}, {'entity': 'bilirubin', 'label': 'TEST', 'start': 1275, 'end': 1284}, {'entity': 'bilirubin', 'label': 'TEST', 'start': 1845, 'end': 185